In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from transformers import AutoModel

**Reward Model Architecture**

In [2]:
class RewardModel(nn.Module):
    def __init__(self, base_model_name, hidden_size=768):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(base_model_name)
        
        self.reward_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )
    
    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        
        if hasattr(outputs, "pooler_output"):
            hidden = outputs.pooler_output
        else:
            hidden = (outputs.last_hidden_state * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(1, keepdim=True)
        
        reward = self.reward_head(hidden)
        return reward.squeeze(-1)

In [ ]:
def train_reward_model(reward_model, preference_data, num_epochs=3):
    optimizer = optim.AdamW(reward_model.parameters(), lr=1e-5)
    
    for epoch in range(num_epochs):
        